In [3]:
using DelimitedFiles
include("open_optimization_problem.jl")   # pulls in the full include chain

n      = 3
J      = fill(1/4, n - 1)
gammas = fill(J[1]/4, n)
tlist  = range(0, 25; length = 100)
excited = ["0"]
ks     = [2, 8, 15]
dissipation = false

cutoff, maxdim = 0.0, 16     # was 1e-10, 200 — maxdim=16 is already exact for n=3
order     = 2     # order of the product formulas being combined
k_ref     = 100   # fine reference standing in for e^{tL} in F_ex
order_ref = 4                # was 2

lsites = liouville_siteinds(n)
rho0   = vectorized_initial_state_mps(lsites, excited)

coeffs = zeros(Float64, length(tlist), length(ks))

for (i, t) in enumerate(tlist)
    if t <= 0
        coeffs[i, :] .= NaN
        continue
    end
    M, _ = open_gram_matrix(n, J, gammas, t, ks, lsites, rho0;
                            cutoff = cutoff, maxdim = maxdim,
                            order = order, dissipation = dissipation)
    L, _ = open_L_vector(n, J, gammas, t, ks, k_ref, lsites, rho0;
                         cutoff = cutoff, maxdim = maxdim,
                         order = order, order_ref = order_ref,
                         dissipation = dissipation)
    c, _ = dynamic_mpf_coefficients(M, L)
    coeffs[i, :] .= c
    println("t = ", round(t, digits = 4), "  c = ", c,
            "  sum = ", sum(c), "  cond(M) = ", cond(M))
end

open("n_3_mpf_coefficients_3815.txt", "w") do io
    println(io, "# t\tc_k3\tc_k8\tc_k12")
    writedlm(io, hcat(collect(tlist), coeffs))
end

t = 0.2525  c = [-0.05426669618546612, 0.794668357798838, 0.25959833838662805]  sum = 1.0  cond(M) = 5.668248381680867e13
t = 0.5051  c = [0.06495351108870892, -1.8244376784064897, 2.759484167317781]  sum = 1.0000000000000002  cond(M) = 2.564078038232262e14
t = 0.7576  c = [0.8487336621452592, -19.047687456951316, 19.19895379480606]  sum = 1.0000000000000036  cond(M) = 9.267112782859526e14
t = 1.0101  c = [-0.011989336487698003, -0.13394481465785338, 1.1459341511455514]  sum = 1.0  cond(M) = 3.2383612790224066e13
t = 1.2626  c = [0.0069132528744137774, -0.5495050059438189, 1.542591753069405]  sum = 1.0  cond(M) = 6.016831560402756e12
t = 1.5152  c = [0.0011664600790960676, -0.42312943043255025, 1.4219629703534542]  sum = 1.0  cond(M) = 8.169138671032883e11
t = 1.7677  c = [0.001237017406719784, -0.42467880017505016, 1.4234417827683303]  sum = 1.0  cond(M) = 1.5575163501762637e11
t = 2.0202  c = [0.0012161496720434644, -0.42421452905396284, 1.4229983793819194]  sum = 1.0  cond(M) = 3.77